<h3 style="color:#1E3A8A; font-family:Arial, Helvetica, sans-serif; margin-bottom:6px;">
  <b>IST 691 — Phase 1 (Part 3): LLM Forward-Looking Extraction</b>
</h3>

<div style="font-family:Arial, Helvetica, sans-serif; line-height:1.55;">
  <p style="margin-top:0;">
    <b>Big picture.</b> This module performs forward-looking belief extraction using a local LLM interface. Each input row is treated as a coherent text unit, and the output is a belief-focused dataset containing normalized sentences and associated soft-label fields for downstream embedding and time-series analysis.
  </p>

  <p>
    <b>Single source of truth (input).</b> The module consumes the chunked transcript dataset produced in Phase 1, Part 2. Each record includes a stable identifier (<code>hash_id</code>) and a text chunk (<code>text</code>) designed to preserve context via overlap and sentence-aware segmentation.
  </p>

  <p>
    <b>Core logic flow.</b>
    <ol style="margin-top:4px;">
      <li><b>Load</b> the chunked transcript dataset and apply an optional limit for controlled execution.</li>
      <li><b>Submit</b> each chunk to the local LLM interface using a strict prompt requiring JSON-only output.</li>
      <li><b>Extract</b> a JSON array of belief objects from the response while tolerating common formatting artifacts.</li>
      <li><b>Normalize</b> each belief into a grammatically complete sentence beginning with <code>Apple</code> or <code>Apple's</code> and resolve pronouns when possible.</li>
      <li><b>Attach soft labels</b> (<code>strategic_focus</code>, <code>temporal_framing</code>, <code>certainty_level</code>, <code>market_position</code>) and preserve an optional <code>speaker</code> field when present.</li>
      <li><b>Export</b> consolidated extraction results and write checkpoint state for reproducible batch execution.</li>
    </ol>
  </p>

  <p>
    <b>Extraction contract.</b> The model returns either (i) a JSON array of belief objects following a fixed schema, or (ii) the exact token <code>NONE</code> when no valid belief content exists. Beliefs are forward-looking, opinion-oriented propositions and exclude pure factual statements or short fragments.
  </p>

  <p>
    <b>Deliverables (local exports).</b>
    <ul style="margin-top:4px;">
      <li><b>Belief extraction results</b>: aggregated belief strings and execution routing metadata keyed by <code>hash_id</code>.</li>
      <li><b>Checkpoint state</b>: batch completion tracking for resumable runs and auditability.</li>
      <li><b>Logs</b>: timestamped execution logs written to file and mirrored to console.</li>
    </ul>
  </p>

  <p style="margin-bottom:0;">
    <b>Console exhibit.</b> The run prints configuration parameters, batch-level progress, processing rate, estimated remaining time, and total runtime at completion.
  </p>
</div>

<hr style="border:none; border-top:1px solid #ddd; margin:12px 0;">


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
====================================================================================================
IST 691 — Phase 1 (Part 3)  |  LLM Forward-Looking Extraction
====================================================================================================

purpose
  - extract forward-looking, belief-oriented statements from Phase 1, Part 2 chunked text units
  - normalize each belief into a standalone sentence beginning with 'Apple' or 'Apple's'
  - attach soft-label fields to each extracted belief for downstream analysis and embedding readiness

inputs (single source of truth)
  - chunked transcript dataset produced in Phase 1, Part 2
  - required columns: hash_id, text
  - each row represents a sentence-aware, overlapping chunk to be analyzed as a whole

processing (logic flow)
  1) load chunked transcript dataset and apply optional record limit for controlled runs
  2) submit each text chunk to the local LLM interface (Ollama) using a strict JSON-only prompt
  3) parse model responses by extracting a JSON array of belief objects; tolerate malformed responses
  4) validate belief objects:
       - require non-empty 'sentence'
       - discard malformed placeholder fragments
       - preserve soft-label fields (strategic_focus, temporal_framing, certainty_level, market_position)
       - preserve optional speaker field when present
  5) aggregate beliefs per originating hash_id and export consolidated extraction results
  6) checkpoint batch progress and log execution details for traceability

outputs (local exports)
  - belief extraction results dataset:
       - hash_id
       - extracted_beliefs (serialized aggregation for each input row)
       - gpu_assigned (execution routing indicator)
  - checkpoint file for batch completion state
  - timestamped execution log (console + file)

console exhibit
  - run configuration parameters (model, temperature, batch size, workers, limit)
  - batch-level progress reporting (rate, elapsed time, estimated remaining time)
  - completion summary and total runtime

====================================================================================================

December 2025 | Syracuse University | IST 691 Deep Learning Term Project

Dujun; Yifeng; Isha
"""


################################################################################
# SECTION 2: PARAMETER SETTINGS & SYSTEM CONFIGURATION
################################################################################
# Core Parameter Settings (adjust these parameters if needed)
MODEL_NAME      = "llama3"      # Model: llama3
TEMPERATURE     = 0.0           # Temperature: 0.0 for deterministic output
BATCH_SIZE      = 300           # Process 300 records per batch
MAX_WORKERS     = 22            # Use up to 22 parallel workers (threads/GPUs)
LIMIT           = 140236          # For testing: process only the first 1000 records

# Directory and File Settings
OUTPUT_DIR      = "./output"
LOG_DIR         = os.path.join(OUTPUT_DIR, "logs")
BATCH_DIR       = os.path.join(OUTPUT_DIR, "batches")
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "checkpoint.json")
FINAL_OUTPUT    = os.path.join(OUTPUT_DIR, "belief_extraction_results_full.csv")
# Create necessary directories if they do not exist
for directory in [OUTPUT_DIR, LOG_DIR, BATCH_DIR]:
    if not os.path.exists(directory):
        os.makedirs(directory)

################################################################################
# SECTION 3: LOGGING SETUP
################################################################################
import sys
import csv
import json
import time
import logging
import re
import os
import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from typing import Dict, Any, List, Optional
import argparse

# Configure logging to log to both console and a timestamped log file
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger()
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
file_handler = logging.FileHandler(os.path.join(LOG_DIR, f"process_log_{timestamp}.log"))
file_handler.setLevel(logging.DEBUG)
file_formatter = logging.Formatter('%(asctime)s [%(levelname)s] - %(message)s')
file_handler.setFormatter(file_formatter)
logger.addHandler(file_handler)

################################################################################
# SECTION 4: REQUIRED LIBRARY IMPORTS
################################################################################
try:
    import ollama
except ImportError:
    print("[ERROR] The 'ollama' library is not installed. Please install it with: pip3 install ollama")
    sys.exit(1)

try:
    import torch
except ImportError:
    # Dummy implementation if torch is not available.
    class DummyTorch:
        def cuda(self):
            return self
        def device_count(self):
            return 1
        def is_available(self):
            return False
    torch = DummyTorch()

################################################################################
# SECTION 5: FULL CLEANUP FUNCTION
################################################################################
def full_cleanup():
    """
    Fully clears previous run data:
      - Deletes checkpoint files.
      - Deletes the batches directory.
      - Deletes the final output CSV file.
    This ensures that the next run starts from a completely fresh state.
    """
    import shutil
    cleanup_paths = [
        CHECKPOINT_FILE,
        os.path.join(OUTPUT_DIR, "belief_extraction_checkpoint.json"),
        BATCH_DIR,
        FINAL_OUTPUT
    ]
    for path in cleanup_paths:
        if os.path.isdir(path):
            try:
                shutil.rmtree(path)
                logger.info(f"Deleted directory: {path}")
            except Exception as e:
                logger.warning(f"Could not delete directory {path}: {e}")
        elif os.path.isfile(path):
            try:
                os.remove(path)
                logger.info(f"Deleted file: {path}")
            except Exception as e:
                logger.warning(f"Could not delete file {path}: {e}")
        else:
            logger.info(f"Path not found (skipped): {path}")
    logger.info("=== Full Cleanup Process Completed ===")

################################################################################
# SECTION 6: GPU MONITORING FUNCTION
################################################################################
def log_gpu_usage(num_gpus: int):
    """
    Logs the memory usage and properties of available GPUs.
    """
    if hasattr(torch, "cuda") and callable(torch.cuda.device_count) and torch.cuda.device_count() > 0:
        for i in range(num_gpus):
            try:
                props = torch.cuda.get_device_properties(i)
                logger.info(f"  GPU {i}: {props.name} ({props.total_memory / (1024**3):.2f} GB)")
            except Exception:
                logger.info(f"  GPU {i}: Available")
    else:
        logger.info("No GPU available; running on CPU.")

################################################################################
# SECTION 7: OLLAMA EXTRACTION FUNCTIONS
################################################################################
def extract_content_from_ollama_response(result: Any) -> str:
    """
    Given the raw result from ollama.chat(...), extract the plain text.
    """
    if hasattr(result, "message") and hasattr(result.message, "content"):
        return result.message.content
    elif isinstance(result, dict) and "message" in result:
        msg = result["message"]
        if isinstance(msg, dict) and "content" in msg:
            return msg["content"]
        elif hasattr(msg, "content"):
            return msg.content
        else:
            return str(msg)
    elif isinstance(result, str):
        return result
    else:
        response_str = str(result)
        content_marker = "content='"
        if content_marker in response_str:
            start_idx = response_str.find(content_marker) + len(content_marker)
            end_idx = response_str.find("'", start_idx)
            if start_idx < end_idx:
                return response_str[start_idx:end_idx]
        quote_matches = re.findall(r"'([^']*content[^']*)'", response_str)
        if quote_matches:
            return quote_matches[0]
        return response_str

def extract_json_array(text: str) -> Optional[List[dict]]:
    """
    Looks for and extracts a JSON array embedded in text.
    Returns an empty list if a known malformed fragment (just "sentence") is encountered.
    """
    text = text.strip()
    if not text:
        return None
    if text == '\n "sentence"' or text.strip() == '"sentence"':
        logger.warning("Found malformed 'sentence' fragment - returning empty list")
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = json.loads(text)
            if isinstance(parsed, list):
                return parsed
        except json.JSONDecodeError:
            pass
    start_idx = text.find("[")
    end_idx = text.rfind("]")
    if start_idx != -1 and end_idx != -1 and start_idx < end_idx:
        bracketed = text[start_idx:end_idx+1]
        try:
            parsed = json.loads(bracketed)
            if isinstance(parsed, list):
                return parsed
        except json.JSONDecodeError:
            pass
    json_pattern = r'\[\s*{.*?}\s*(?:,\s*{.*?}\s*)*\]'
    matches = re.findall(json_pattern, text, re.DOTALL)
    for match in matches:
        try:
            parsed = json.loads(match)
            if isinstance(parsed, list):
                return parsed
        except json.JSONDecodeError:
            continue
    json_indicators = ["```json", "```", "JSON array:", "JSON:"]
    for indicator in json_indicators:
        if indicator in text:
            parts = text.split(indicator, 1)
            if len(parts) > 1:
                json_candidate = parts[1].strip()
                if "```" in json_candidate:
                    json_candidate = json_candidate.split("```")[0].strip()
                start_idx = json_candidate.find("[")
                end_idx = json_candidate.rfind("]")
                if start_idx != -1 and end_idx != -1 and start_idx < end_idx:
                    bracketed = json_candidate[start_idx:end_idx+1]
                    try:
                        parsed = json.loads(bracketed)
                        if isinstance(parsed, list):
                            return parsed
                    except json.JSONDecodeError:
                        continue
    return None

def extract_beliefs_from_paragraph(paragraph: str, model_name: str = MODEL_NAME,
                                   temperature: float = TEMPERATURE, max_retries: int = 3,
                                   retry_delay: float = 2.0) -> List[Dict[str, Any]]:
    """
    Sends a paragraph to the Ollama model and extracts belief statements.
    Expects a JSON array of belief objects with keys:
        "sentence", "strategic_focus", "temporal_framing", "certainty_level", "market_position"
    An optional "speaker" field may be provided.
    """
    paragraph = paragraph.strip()
    if not paragraph:
        return []
    prompt = PROMPT_TEMPLATE.format(paragraph=paragraph)
    
    logger.info("Calling Ollama API for paragraph extraction...")
    
    response_text = ""
    for attempt in range(max_retries):
        try:
            result = ollama.chat(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                options={"temperature": temperature}
            )
            response_text = extract_content_from_ollama_response(result)
            break
        except Exception as exc:
            logger.error(f"[extract_beliefs_from_paragraph] Attempt {attempt+1} error: {exc}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                return []
    response_text = response_text.strip()
    if not response_text or response_text.upper() == "NONE":
        return []
    if response_text == '\n "sentence"' or response_text.strip() == '"sentence"':
        logger.warning("Received problematic 'sentence' fragment response - ignoring")
        return []
    arr = extract_json_array(response_text)
    if arr is not None:
        beliefs = []
        for item in arr:
            if not isinstance(item, dict):
                continue
            sentence = item.get("sentence", "").strip()
            if sentence.lower() == "sentence" or not sentence:
                continue
            sfocus = item.get("strategic_focus", "").strip()
            tframe = item.get("temporal_framing", "").strip()
            clevel = item.get("certainty_level", "").strip()
            mpos = item.get("market_position", "").strip()
            speaker = item.get("speaker", "").strip()  # Optional field
            beliefs.append({
                "sentence": sentence,
                "strategic_focus": sfocus,
                "temporal_framing": tframe,
                "certainty_level": clevel,
                "market_position": mpos,
                "speaker": speaker
            })
        return beliefs
    # Fallback: Attempt line-by-line extraction if JSON is not found.
    beliefs = []
    lines = response_text.split('\n')
    for line in lines:
        line_clean = line.strip().strip('"').strip()
        if not line_clean or line_clean.lower() == "sentence":
            continue
        if "BELIEF:" in line_clean:
            belief_text = line_clean[line_clean.find("BELIEF:") + 7:].strip()
            if not belief_text:
                continue
            strategic_focus = "Product-Centric"
            temporal_framing = "Near-Term"
            certainty_level = "Medium"
            market_position = "Neutral"
            lower_text = belief_text.lower()
            if any(term in lower_text for term in ["ecosystem", "platform", "integration"]):
                strategic_focus = "Ecosystem-Centric"
            elif any(term in lower_text for term in ["ai", "intelligence", "machine learning"]):
                strategic_focus = "Artificial Intelligence Focus"
            elif any(term in lower_text for term in ["customer", "user", "experience"]):
                strategic_focus = "Customer Experience Focus"
            elif any(term in lower_text for term in ["market", "industry", "business", "revenue"]):
                strategic_focus = "Market/Business Focus"
            if any(term in lower_text for term in ["long term", "future", "years"]):
                temporal_framing = "Long-Term"
            elif any(term in lower_text for term in ["coming months", "this year"]):
                temporal_framing = "Mid-Term"
            if any(term in lower_text for term in ["will", "definitely", "certainly"]):
                certainty_level = "High"
            elif any(term in lower_text for term in ["may", "might", "could", "possibly"]):
                certainty_level = "Low"
            if any(term in lower_text for term in ["compete", "outperform", "lead", "revolutionary"]):
                market_position = "Aggressive"
            elif any(term in lower_text for term in ["protect", "maintain", "preserve"]):
                market_position = "Defensive"
            if not belief_text.startswith("Apple") and not belief_text.startswith("Apple's"):
                belief_text = f"Apple {belief_text}"
            beliefs.append({
                "sentence": belief_text,
                "strategic_focus": strategic_focus,
                "temporal_framing": temporal_framing,
                "certainty_level": certainty_level,
                "market_position": market_position,
                "speaker": ""
            })
    return beliefs

################################################################################
# SECTION 8: DOCUMENT PROCESSING & CSV I/O FUNCTIONS
################################################################################
def process_document_beliefs(row: Dict[str, Any], gpu_id: int) -> Dict[str, Any]:
    """
    Processes a CSV row by extracting beliefs from its 'text' column.
    Returns a dictionary with keys: "hash_id", "extracted_beliefs", and "gpu_assigned".
    """
    try:
        text = row.get("text", "").strip()
        row_id = row.get("hash_id", "unknown")
        if not text:
            return {"hash_id": row_id, "extracted_beliefs": "", "gpu_assigned": gpu_id}
        beliefs_list = extract_beliefs_from_paragraph(text, model_name=MODEL_NAME, temperature=TEMPERATURE)
        if beliefs_list:
            belief_strs = []
            for b in beliefs_list:
                single_line = f"{b['sentence']} (Focus={b['strategic_focus']}, Time={b['temporal_framing']}, Cert={b['certainty_level']}, Pos={b['market_position']})"
                belief_strs.append(single_line)
            combined = " || ".join(belief_strs)
        else:
            combined = ""
        return {"hash_id": row_id, "extracted_beliefs": combined, "gpu_assigned": gpu_id}
    except Exception as e:
        logger.error(f"Error processing row: {e}")
        return {"hash_id": row.get("hash_id", "unknown"), "extracted_beliefs": "", "gpu_assigned": gpu_id}

def read_csv_file(csv_path: str) -> List[Dict[str, Any]]:
    """
    Reads a CSV file into a list of dictionaries using the built-in csv module.
    """
    rows = []
    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                rows.append(row)
        logger.info(f"Read {len(rows)} rows from {csv_path}")
    except Exception as e:
        logger.error(f"Error reading CSV file {csv_path}: {e}")
    return rows

def write_csv_file(csv_path: str, data: List[Dict[str, Any]], fieldnames: Optional[List[str]] = None) -> None:
    """
    Writes a list of dictionaries to a CSV file.
    """
    if not data:
        logger.warning("No data to write.")
        return
    if fieldnames is None:
        fieldnames = list(data[0].keys())
    try:
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data)
        logger.info(f"Wrote {len(data)} rows to {csv_path}")
    except Exception as e:
        logger.error(f"Error writing to CSV file {csv_path}: {e}")

def process_in_batches(rows: List[Dict[str, Any]], batch_size: int = BATCH_SIZE,
                       checkpoint_file: str = CHECKPOINT_FILE,
                       max_workers: int = MAX_WORKERS) -> List[Dict[str, Any]]:
    """
    Processes rows in batches from scratch (no checkpoint resume) with detailed
    progress logging and GPU usage monitoring.
    """
    results_data = []
    start_idx = 0  # Always start from 0 for a fresh run
    total_rows = len(rows)
    total_batches = (total_rows - start_idx + batch_size - 1) // batch_size
    num_gpus = max(1, torch.cuda.device_count() if (hasattr(torch, "cuda") and callable(torch.cuda.device_count)) else 1)
    logger.info(f"Using {num_gpus} GPU(s) for processing.")
    overall_start_time = time.time()
    
    for batch_num in range(total_batches):
        batch_start = start_idx + batch_num * batch_size
        batch_end = min(batch_start + batch_size, total_rows)
        logger.info(f"Processing batch {batch_num+1}/{total_batches} (rows {batch_start}-{batch_end-1})")
        batch_start_time = time.time()
        
        # Prepare tasks with round-robin GPU assignment.
        tasks = []
        for i in range(batch_start, batch_end):
            row = rows[i]
            gpu_id = i % num_gpus
            tasks.append((row, gpu_id))
        
        log_gpu_usage(num_gpus)
        
        batch_results = []
        with ThreadPoolExecutor(max_workers=num_gpus) as executor:
            future_to_task = {executor.submit(process_document_beliefs, task[0], task[1]): task for task in tasks}
            for future in tqdm(as_completed(future_to_task), total=len(tasks),
                               desc=f"Batch {batch_num+1}/{total_batches}"):
                try:
                    result = future.result()
                    batch_results.append(result)
                except Exception as exc:
                    logger.error(f"Error processing row: {exc}")
                    task = future_to_task[future]
                    row_id = task[0].get("hash_id", "unknown")
                    batch_results.append({
                        "hash_id": row_id,
                        "extracted_beliefs": "",
                        "gpu_assigned": task[1]
                    })
        results_data.extend(batch_results)
        
        # Update checkpoint after batch processing
        with open(checkpoint_file, "w") as f:
            json.dump({
                "last_processed_idx": batch_end - 1,
                "results_data": results_data,
                "timestamp": time.time()
            }, f)
        
        batch_duration = time.time() - batch_start_time
        items_processed = batch_end - batch_start
        items_per_second = items_processed / batch_duration if batch_duration > 0 else 0
        remaining_batches = total_batches - batch_num - 1
        estimated_remaining_time = remaining_batches * batch_duration
        
        logger.info(f"Completed batch {batch_num+1}/{total_batches} in {batch_duration:.2f} seconds")
        logger.info(f"Processed {batch_end}/{total_rows} documents so far")
        logger.info(f"Processing rate: {items_per_second:.2f} items/second")
        logger.info(f"Estimated time remaining: {str(datetime.timedelta(seconds=int(estimated_remaining_time)))}")
    
    overall_duration = time.time() - overall_start_time
    logger.info(f"Total processing time: {str(datetime.timedelta(seconds=int(overall_duration)))}")
    return results_data

def run_belief_extraction(input_csv: str, output_csv: str, text_col: str = "text", id_col: str = "hash_id", limit: Optional[int] = LIMIT) -> None:
    """
    Main pipeline:
      1. Reads the input CSV (must contain 'hash_id' and 'text' columns).
      2. If a limit is provided, processes only the first N records.
      3. Extracts beliefs using the Ollama API.
      4. Writes aggregated results to the output CSV.
    """
    data = read_csv_file(input_csv)
    if not data:
        logger.error("No data loaded; exiting.")
        return
    if limit is not None:
        data = data[:limit]
        logger.info(f"Processing only the first {limit} records.")
    logger.info(f"Loaded {len(data)} records from {input_csv}. Starting extraction...")
    results = process_in_batches(data, batch_size=BATCH_SIZE)
    write_csv_file(output_csv, results, fieldnames=[id_col, "extracted_beliefs", "gpu_assigned"])
    logger.info("Belief extraction completed successfully!")

################################################################################
# SECTION 9: PROMPT TEMPLATE DEFINITION
################################################################################
PROMPT_TEMPLATE = (
    "You are provided with a paragraph from an Apple executive transcript.\n"
    "Your task is to analyze the entire paragraph in context and extract ONLY the belief-related content. "
    "Do NOT output the full paragraph.\n\n"
    
    "A valid belief is defined as a forward-looking, opinion-based proposition that:\n"
    "• Predicts or projects a future event, trend, or outcome.\n"
    "• Expresses explicit or implicit conviction using phrases such as \"we believe\", \"we expect\", \"we will\", or \"our goal is\".\n"
    "• Is not a mere factual statement (e.g., \"the tree is green\", \"the house is far away\", or \"tomorrow is the first day of the month\" should be ignored).\n"
    "• Is not a simple greeting, an isolated name, or a single phrase (e.g., \"Brian, thank you\" or \"my boss\").\n"
    "• Contains at least 10 words.\n\n"
    
    "Instructions:\n"
    "1. Process the paragraph as a whole without manually splitting it. Use context to decide if a sentence expresses a forward-looking opinion.\n"
    "2. For each valid belief found:\n"
    "   a. Normalize it into a grammatically complete sentence beginning with 'Apple' or 'Apple's' (replace pronouns appropriately).\n"
    "   b. If the belief spans multiple sentences and implies a speaker (e.g., an Apple executive), include a field 'speaker' with that name; otherwise, use 'Speaker' or 'Apple'.\n"
    "   c. Resolve pronouns to their proper nouns where possible. Do not include any other text or explanations.\n"
    "   d. Convert it into the following JSON structure:\n\n"
    "      [\n"
    "        {{\n"
    "          \"sentence\": \"<extracted and normalized belief sentence>\",\n"
    "          \"speaker\": \"<Apple or Speaker or speaker's name>\",\n"
    "          \"strategic_focus\": \"<choose one: Product-Centric, Ecosystem-Centric, Artificial Intelligence Focus, Customer Experience Focus, Market/Business Focus>\",\n"
    "          \"temporal_framing\": \"<choose one: Near-Term, Mid-Term, Long-Term>\",\n"
    "          \"certainty_level\": \"<choose one: High, Medium, Low>\",\n"
    "          \"market_position\": \"<choose one based on these rules>\"\n"
    "        }},\n"
    "        ...\n"
    "      ]\n\n"
    "   For 'market_position', apply the following explicit rules:\n"
    "     - If the belief contains any of the keywords 'protect', 'maintain', or 'preserve', assign 'Defensive'.\n"
    "     - Else if the belief contains any of the keywords 'compete', 'outperform', 'lead', or 'revolutionary', assign 'Aggressive'.\n"
    "     - Otherwise, assign 'Neutral'.\n\n"
    "3. Process the paragraph sequentially so that once a belief sentence is extracted, overlapping content is not re-extracted.\n"
    "4. If no belief-related content is found, output exactly \"NONE\".\n\n"
    "Do NOT include any other commentary or text beyond this JSON output.\n"
    "\nParagraph:\n\"\"\"\n{paragraph}\n\"\"\"\n"
)

################################################################################
# SECTION 10: MAIN EXECUTION & ARGUMENT PARSING
################################################################################
if __name__ == "__ipykernel_launcher__" or __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Extract forward-looking belief statements using the Ollama API with built-in modules only."
    )
    parser.add_argument("--input", "-i", required=True, help="Path to input CSV (must contain 'hash_id' and 'text' columns).")
    parser.add_argument("--output", "-o", required=True, help="Path to output CSV file for extracted beliefs.")
    parser.add_argument("--model", "-m", default=MODEL_NAME, help="Ollama model name (default: llama3).")
    parser.add_argument("--log-level", default="INFO", help="Logging level (DEBUG, INFO, etc.).")
    parser.add_argument("--batch_size", type=int, default=BATCH_SIZE, help="Batch size (default: 300).")
    parser.add_argument("--max_workers", type=int, default=MAX_WORKERS, help="Max workers for parallel processing (default: 22).")
    parser.add_argument("--temperature", type=float, default=TEMPERATURE, help="Ollama API temperature (default: 0.0).")
    parser.add_argument("--limit", type=int, default=LIMIT, help="Process only the first N records (default: 1000).")
    
    # Simulate command-line arguments for Jupyter Notebook if running in that environment.
    if "ipykernel_launcher.py" in sys.argv[0]:
        simulated_args = [
            "--input", os.path.join(OUTPUT_DIR, "../Section3_Phase1_Chunked_Transcripts_For_LLM.csv"),
            "--output", FINAL_OUTPUT,
            "--model", MODEL_NAME,
            "--batch_size", str(BATCH_SIZE),
            "--max_workers", str(MAX_WORKERS),
            "--temperature", str(TEMPERATURE),
            "--log-level", "INFO",
            "--limit", str(LIMIT)
        ]
        args = parser.parse_args(simulated_args)
    else:
        args = parser.parse_args()
    
    logger.setLevel(getattr(logging, args.log_level.upper()))
    
    # Print configuration parameters
    print("Configuration Parameters:")
    print(f"  Input CSV           : {args.input}")
    print(f"  Output CSV          : {args.output}")
    print(f"  Model               : {args.model}")
    print(f"  Batch Size          : {args.batch_size}")
    print(f"  Max Workers         : {args.max_workers}")
    print(f"  Temperature         : {args.temperature}")
    print(f"  Limit               : {args.limit if args.limit is not None else 'No limit'}")
    print(f"  Log Level           : {args.log_level}")
    print("-------------------------------------------------------------\n")
    
    # OPTIONAL: Uncomment the following line to perform a full cleanup before extraction.
    # full_cleanup()
    
    start_time = time.time()
    run_belief_extraction(args.input, args.output, text_col="text", id_col="hash_id", limit=args.limit)
    total_duration = time.time() - start_time
    print(f"Script execution completed in {str(datetime.timedelta(seconds=int(total_duration)))}")

Configuration Parameters:
  Input CSV           : ./output/../Section3_Phase1_Chunked_Transcripts_For_LLM.csv
  Output CSV          : ./output/belief_extraction_results_full.csv
  Model               : llama3
  Batch Size          : 300
  Max Workers         : 22
  Temperature         : 0.0
  Limit               : 140236
  Log Level           : INFO
-------------------------------------------------------------



[INFO] Read 140236 rows from ./output/../Section3_Phase1_Chunked_Transcripts_For_LLM.csv
[INFO] Processing only the first 140236 records.
[INFO] Loaded 140236 records from ./output/../Section3_Phase1_Chunked_Transcripts_For_LLM.csv. Starting extraction...
[INFO] Using 4 GPU(s) for processing.
[INFO] Processing batch 1/468 (rows 0-299)
[INFO]   GPU 0: Tesla V100-PCIE-16GB (15.77 GB)
[INFO]   GPU 1: Tesla V100-PCIE-16GB (15.77 GB)
[INFO]   GPU 2: Tesla V100-PCIE-16GB (15.77 GB)
[INFO]   GPU 3: Tesla V100-PCIE-16GB (15.77 GB)
[INFO] Calling Ollama API for paragraph extraction...
[INFO] Calling Ollama API for paragraph extraction...
[INFO] Calling Ollama API for paragraph extraction...
Batch 1/468:   0%|                                                                                                                                                                                                                                                                                                   

Thought for a couple of seconds


<div style="border: 2px solid #1A5276; border-radius: 8px; padding: 18px; background: linear-gradient(to bottom right, #FDFEFE, #FCF3CF); font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 14px; color: #1A1A1A; line-height: 1.7; max-width: 1000px; box-sizing: border-box;">
  <p style="font-size: 23px; font-weight: 700; color: #154360; margin-bottom: 19px;">
    Step 3 — Final Takeaway: Belief Extraction &amp; Classification Impact
  </p>
  <p>
    This step converts cleaned transcript paragraphs into fully labeled belief statements, embedding strategic, temporal, certainty, and market metadata to fuel robust downstream analyses.
  </p>
  <hr style="border: none; border-top: 1px solid #D5D8DC; margin: 16px 0;">

  <p style="font-weight: 600; color:#1A5276;">Impact &amp; Transition:</p>
  <ul style="margin-top: 6px; padding-left: 20px;">
    <li><strong>Structured belief dataset:</strong> Transforms unstructured text into JSON‐formatted records ready for clustering and modeling.</li>
    <li><strong>Feature enrichment:</strong> Metadata fields (<code>strategic_focus</code>, <code>temporal_framing</code>, etc.) become engineered predictors for forecasting.</li>
    <li><strong>Pipeline continuity:</strong> Checkpointing and logging ensure fault‑tolerant execution, smoothing the handoff to Session 4 topic modeling.</li>
    <li><strong>Downstream readiness:</strong> Extracted beliefs directly feed into semantic clustering, trend analysis, and predictive workflows in the next phase.</li>
  </ul>

  <p style="font-weight: 600; color:#CC0000;">Summary:</p>
  <p style="color:#CC0000;">
    By rigorously extracting and classifying forward‑looking beliefs, this phase establishes the critical data foundation for semantic clustering and forecasting—ensuring the next session can build on a rich, reliable feature set.
  </p>
</div>
